# Ingestion Pipeline

End-to-end pipeline from raw PubMed abstracts to populated vector databases.

**Step 1 — Load & Chunk:** Load raw `.txt` files via LangChain DirectoryLoader, filter out multilingual abstracts, strip headers/footers, and chunk with RecursiveCharacterTextSplitter.  
**Step 2 — Ingest:** Push chunks into multiple vector databases (ChromaDB, Qdrant, LanceDB, Weaviate).

**Input:** `data/raw/{pubid}.txt`  
**Output:** Populated vector stores in `vectorstores/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install langchain-huggingface
!pip install langchain-chroma
!pip install qdrant-client langchain-qdrant
!pip install faiss-cpu
!pip install lancedb
!pip install -U weaviate-client langchain-weaviate
!pip install langdetect
!pip install -U langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [18]:
import sys
sys.path.append("..")

import os
import re
from pathlib import Path
from langdetect import detect, LangDetectException
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd
import config
import json
import uuid
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "true"
from google.colab import userdata

In [9]:

import importlib
importlib.reload(config)

<module 'config' from '/content/config.py'>

In [ ]:
config.CHUNK_SIZE

800

## Parsing & Cleaning

Two minimal operations:
- **`detect_multilingual`** — flags abstracts with a non-English `Publisher:` translation block (common in bilingual journals like Canadian ones). These are excluded from the KB.
- **`extract_abstract_body`** — strips the PubMed header (citation line, title, authors, affiliations) and footer (DOI, PMID, translated blocks), leaving only the abstract text.

In [10]:
FOOTER_PATTERN = re.compile(
    r"\n(DOI:|PMID:|PMCID:|Published by|Publisher:|Copyright).*",
    re.DOTALL,
)


def detect_multilingual(raw_text):
    """Return True if the abstract has a non-English Publisher: translation block."""
    publisher_match = re.search(r"^Publisher:", raw_text, re.MULTILINE)
    if not publisher_match:
        return False
    excerpt = raw_text[publisher_match.end() : publisher_match.end() + 400].strip()
    if not excerpt:
        return False
    try:
        return detect(excerpt) != "en"
    except LangDetectException:
        return False


def extract_abstract_body(raw_text):
    """Strip PubMed header and footer; return only the abstract body text."""
    # Cut at the first footer marker
    text = FOOTER_PATTERN.sub("", raw_text).strip()

    paragraphs = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]

    body_start = len(paragraphs)
    for i, para in enumerate(paragraphs):
        # Structured abstract: BACKGROUND:, METHODS:, RESULTS:, etc.
        if re.match(r"[A-Z][A-Z ]+:", para):
            body_start = i
            break
        # Unstructured abstract with explicit heading
        if para.strip().lower() == "abstract":
            body_start = i + 1
            break
        # After the 3-para header (citation / title / authors),
        # the first long paragraph that isn't an affiliations block
        if i >= 3 and len(para) > 150 and not re.search(r"\(\d+\)", para[:80]):
            body_start = i
            break

    return "\n".join(paragraphs[body_start:])

In [ ]:
extract_abstract_body("""
1. BMJ. 1989 Jun 24;298(6689):1684-6. doi: 10.1136/bmj.298.6689.1684.

Inhibin: a new circulating marker of hydatidiform mole?

Yohkaichiya T(1), Fukaya T, Hoshiai H, Yajima A, de Kretser DM.

Author information:
(1)Department of Anatomy, Monash University, Clayton, Australia.

OBJECTIVE: To define the concentrations of inhibin in serum and tissue of
patients with hydatidiform mole and assess their value as a clinical marker of
the condition.
DESIGN: Prospective study of new patients with hydatidiform mole, comparison of
paired observations, and case-control analysis.
SETTING: A university hospital, two large public hospitals, and a private
women's clinic in Japan.
PATIENTS: Seven consecutive referred patients seen over four months with newly
diagnosed complete hydatidiform mole, including one in whom the mole was
accompanied by viable twin fetuses (case excluded from statistical analysis
because of unique clinical features). All patients followed up for six months
after evacuation of molar tissue.
END POINT: Correlation of serum inhibin concentrations with trophoblastic
disease.
MEASUREMENTS AND MAIN RESULTS: Serum concentrations of inhibin, human chorionic
gonadotrophin, and follicle stimulating hormone were compared before and seven
to 10 days after evacuation of the mole. Before evacuation the serum inhibin
concentrations (median 8.3 U/ml; 95% confidence interval 2.4 to 34.5) were
significantly greater than in 21 normal women at the same stage of pregnancy
(2.8 U/ml; 2.1 to 3.6), and inhibin in molar tissue was also present in high
concentrations (578 U/ml cytosol; 158 to 1162). Seven to 10 days after
evacuation inhibin concentrations in serum samples from the same patients
declined significantly to values (0.4 U/ml; 0.1 to 1.4) similar to those seen in
the follicular phase of normal menstrual cycles. None of the four patients whose
serum inhibin concentrations were 0.4 U/ml or less after evacuation developed
persistent trophoblastic disease. Though serum human chorionic gonadotrophin
concentrations declined after evacuation (6.6 x 10(3) IU/l; 0.8 x 10(3) to 32.6
x 10(3], they remained far higher than in non-pregnant women. Serum follicle
stimulating hormone concentrations remained suppressed.
CONCLUSIONS: In this small study serum inhibin concentrations higher than those
found in the early follicular phase one to two weeks after evacuation of a
hydatidiform mole seemed to be specific for persistent trophoblastic disease.
Further data are needed to confirm these promising results.

DOI: 10.1136/bmj.298.6689.1684
PMCID: PMC1836764
PMID: 2503176 [Indexed for MEDLINE]
""")

"OBJECTIVE: To define the concentrations of inhibin in serum and tissue of \npatients with hydatidiform mole and assess their value as a clinical marker of \nthe condition.\nDESIGN: Prospective study of new patients with hydatidiform mole, comparison of \npaired observations, and case-control analysis.\nSETTING: A university hospital, two large public hospitals, and a private \nwomen's clinic in Japan.\nPATIENTS: Seven consecutive referred patients seen over four months with newly \ndiagnosed complete hydatidiform mole, including one in whom the mole was \naccompanied by viable twin fetuses (case excluded from statistical analysis \nbecause of unique clinical features). All patients followed up for six months \nafter evacuation of molar tissue.\nEND POINT: Correlation of serum inhibin concentrations with trophoblastic \ndisease.\nMEASUREMENTS AND MAIN RESULTS: Serum concentrations of inhibin, human chorionic \ngonadotrophin, and follicle stimulating hormone were compared before and sev

## Chunking & Loading

`load_and_chunk_abstracts` does everything in one pass: load via LangChain DirectoryLoader → filter multilingual → extract body → chunk. Returns `(documents, ids, multilingual_pubids)`.

In [23]:
def get_text_splitter(chunk_size=None, chunk_overlap=None, separators=None):
    """Return a RecursiveCharacterTextSplitter configured from config defaults or overrides."""
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size or config.CHUNK_SIZE,
        chunk_overlap=chunk_overlap or config.CHUNK_OVERLAP,
        separators=separators or config.CHUNK_SEPARATORS,
    )


def load_and_chunk_abstracts(abstracts_dir, text_splitter=None, mesh_lookup=None):
    """Load all .txt abstracts, filter multilingual, extract body text, and chunk.

    Args:
        abstracts_dir: Directory containing one .txt file per PubMed abstract.
        text_splitter: Optional pre-configured splitter; defaults to get_text_splitter().
        mesh_lookup: Optional dict mapping pubid -> list of MeSH term strings.
                     When provided, each chunk's metadata includes a 'mesh_terms' field.

    Returns:
        documents: List of LangChain Document objects ready for ingestion.
        ids: Parallel list of document IDs (f"{pubid}_chunk{n}").
        multilingual_pubids: PubIDs excluded because they contained a non-English
                             Publisher: translation block.
    """
    if text_splitter is None:
        text_splitter = get_text_splitter()

    loader = DirectoryLoader(
        str(abstracts_dir),
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
        show_progress=False,
        silent_errors=True,
    )
    raw_docs = loader.load()
    print(f"Loaded {len(raw_docs)} abstract files")

    documents = []
    ids = []
    multilingual_pubids = []
    skipped = []
    processed = {"pubid": [], "body": []}

    for doc in raw_docs:
        raw_text = doc.page_content
        pubid = os.path.basename(doc.metadata["source"]).replace(".txt", "")

        if detect_multilingual(raw_text):
            multilingual_pubids.append(pubid)
            continue

        body = extract_abstract_body(raw_text)
        if not body or len(body) < 100:
            skipped.append(pubid)
            continue

        # Adding to processed files
        processed["pubid"].append(pubid)
        processed["body"].append(body)

        mesh_terms = mesh_lookup.get(pubid, []) if mesh_lookup else []
        chunks = text_splitter.split_text(body)
        for chunk_idx, chunk in enumerate(chunks):
            doc_id = f"{pubid}_chunk{chunk_idx}"
            chunk_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, doc_id))
            metadata = {
                "pubid": pubid,
                "source": doc.metadata["source"],
                "chunk_index": chunk_idx,
                "char_count": len(chunk),
                "mesh_terms": mesh_terms,
            }
            documents.append(Document(
                page_content=chunk,
                metadata=metadata,
                id=doc_id,
            ))
            ids.append(chunk_id)

    n_ingested = len(raw_docs) - len(multilingual_pubids) - len(skipped)
    print(f"Created {len(documents)} chunks from {n_ingested} abstracts")
    if multilingual_pubids:
        print(f"Excluded {len(multilingual_pubids)} multilingual abstracts")
    if skipped:
        print(f"Skipped {len(skipped)} abstracts with no extractable body")

    processed_df = pd.DataFrame(processed)
    processed_df.to_csv(config.DATA_PROCESSED_DIR / "processed_abstracts.csv", index=False)

    return documents, ids, multilingual_pubids


def save_knowledge_base(documents, ids, output_dir=None):
    """Serialize the chunked document corpus to a JSON file for reproducibility.

    Saves a list of records, each containing the chunk id, page_content, and all
    metadata fields. This snapshot can be reloaded without re-parsing the raw .txt
    files and serves as the ground-truth corpus for all downstream experiments.

    Args:
        documents: List of LangChain Document objects from load_and_chunk_abstracts.
        ids: Parallel list of document ID strings.
        output_dir: Directory to write knowledge_base.json into;
                    defaults to config.DATA_KNOWLEDGE_BASE_DIR.
    """
    output_dir = Path(output_dir) if output_dir else config.DATA_KNOWLEDGE_BASE_DIR
    output_dir.mkdir(parents=True, exist_ok=True)

    records = [
        {"id": doc_id, "text": doc.page_content, **doc.metadata}
        for doc_id, doc in zip(ids, documents)
    ]

    output_path = output_dir / "knowledge_base.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(records)} chunks to {output_path}")

## Ingestion Functions

One function per vector database. Each is idempotent — skips ingestion if the store already exists.

In [31]:
def ingest_to_chroma(documents, ids, embeddings, db_name="pubmed_chroma"):
    from langchain_chroma import Chroma
    persist_dir = str(config.VECTORSTORE_DIR / db_name)
    print(persist_dir)

    vector_store = Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )

    if vector_store._collection.count() == 0:
        vector_store.add_documents(documents=documents, ids=ids)
        print(f"Total: {len(documents)} chunks ingested into ChromaDB")
    else:
        print(f"Loaded existing ChromaDB from {persist_dir}")

    return vector_store


def ingest_to_qdrant(documents, ids, embeddings, collection_name="pubmed_qdrant", path=None):
    from langchain_qdrant import QdrantVectorStore
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams

    path = path or str(config.VECTORSTORE_DIR / collection_name)

    sample_embedding = embeddings.embed_query("test")
    embedding_dim = len(sample_embedding)

    client = QdrantClient(path=path)

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
        )

    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )

    existing = client.get_collection(collection_name).points_count
    if existing == 0:
        vector_store.add_documents(documents=documents, ids=ids)
        print(f"Ingested {len(documents)} chunks into Qdrant")
    else:
        print(f"Loaded existing Qdrant collection ({existing} points)")

    return vector_store


def ingest_to_lancedb(documents, ids, embeddings, table_name="pubmed_lance", path=None):
    import lancedb

    path = path or str(config.VECTORSTORE_DIR / "lancedb")
    db = lancedb.connect(path)

    texts = [doc.page_content for doc in documents]
    metadatas = [doc.metadata for doc in documents]

    if table_name not in db.table_names():
        vectors = embeddings.embed_documents(texts)
        data = [
            {"id": doc_id, "text": text, "vector": vec, **meta}
            for doc_id, text, vec, meta in zip(ids, texts, vectors, metadatas)
        ]
        table = db.create_table(table_name, data=data)
        print(f"Ingested {len(documents)} chunks into LanceDB")
    else:
        table = db.open_table(table_name)
        print(f"Loaded existing LanceDB table '{table_name}' ({table.count_rows()} rows)")

    return db, table


def ingest_to_faiss(documents, embeddings, index_name="pubmed_faiss", path=None):
    """Ingest documents into a local FAISS index, or load it if it already exists.

    FAISS uses its own internal integer indices so no ids parameter is needed.
    The index is saved as index.faiss + index.pkl under vectorstores/{index_name}/.

    Args:
        documents: LangChain Document objects to index.
        embeddings: LangChain embeddings instance used to encode chunks.
        index_name: Subdirectory name under vectorstores/ for the saved index.
        path: Override the storage path.

    Returns:
        FAISS vector store instance.
    """
    from langchain_community.vectorstores import FAISS

    save_path = path or str(config.VECTORSTORE_DIR / index_name)

    if os.path.exists(save_path) and os.listdir(save_path):
        vector_store = FAISS.load_local(
            save_path, embeddings, allow_dangerous_deserialization=True
        )
        print(f"Loaded existing FAISS index from {save_path} ({len(vector_store.index_to_docstore_id)} vectors)")
    else:
        os.makedirs(save_path, exist_ok=True)
        vector_store = FAISS.from_documents(documents, embeddings)
        vector_store.save_local(save_path)
        print(f"Ingested {len(documents)} chunks into FAISS, saved to {save_path}")

    return vector_store


def ingest_to_weaviate(documents, ids, embeddings, index_name='pubmed_weaviate', api_key=None):
    import weaviate
    from weaviate.classes.init import Auth
    from langchain_weaviate import WeaviateVectorStore

    url = "https://fiuznxyotgul4dwnhuymg.c0.asia-southeast1.gcp.weaviate.cloud"
    api_key = api_key or os.getenv("WEAVIATE_KEY", "")
    api_key = api_key or userdata.get("WEAVIATE_KEY")

    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=url,
        auth_credentials=Auth.api_key(api_key),
    )

    vector_store = WeaviateVectorStore(
        client=client,
        index_name=index_name,
        text_key="text",
        embedding=embeddings,
    )

    vector_store.add_documents(documents=documents, ids=ids)
    print(f"Ingested {len(documents)} chunks into Weaviate")

    return vector_store, client

---
## Step 1: Load, Clean & Chunk

In [25]:
mesh_lookup_path = config.DATA_PROCESSED_DIR / "mesh_lookup.json"
with open(mesh_lookup_path, "r", encoding="utf-8") as f:
    mesh_lookup = json.load(f)
print(f"Loaded MeSH lookup for {len(mesh_lookup)} pubids from {mesh_lookup_path}")

text_splitter = get_text_splitter()
documents, ids, multilingual_pubids = load_and_chunk_abstracts(
    config.DATA_RAW_DIR, text_splitter, mesh_lookup=mesh_lookup
)

save_knowledge_base(documents, ids)

Loaded MeSH lookup for 1000 pubids from /content/data/processed/mesh_lookup.json
Loaded 1000 abstract files
Created 2849 chunks from 999 abstracts
Excluded 1 multilingual abstracts
Saved 2849 chunks to /content/data/knowledge_base/knowledge_base.json


### Multilingual Abstracts Excluded

These pubids had a non-English `Publisher:` block and were skipped. Keep this list if you want to exclude them from evaluation too.

In [26]:
print(f"Multilingual abstracts excluded from KB ({len(multilingual_pubids)} total):")
print(multilingual_pubids)

Multilingual abstracts excluded from KB (1 total):
['24666444']


### Sample Chunks

In [29]:
for doc in documents[:5]:
    print(f"ID:       {doc.id}")
    print(f"Chunk:    {doc.metadata['chunk_index']}")
    print(f"Chars:    {doc.metadata['char_count']}")
    print(f"MeSH:     {doc.metadata['mesh_terms'][:3]}")
    print(f"Content:  {doc.page_content[:200]}")
    print()

ID:       21645374_chunk0
Chunk:    0
Chars:    787
MeSH:     ['Alismataceae', 'Apoptosis', 'Cell Differentiation']
Content:  BACKGROUND: Programmed cell death (PCD) is the regulated death of cells within 
an organism. The lace plant (Aponogeton madagascariensis) produces perforations 
in its leaves through PCD. The leaves o

ID:       21645374_chunk1
Chunk:    1
Chars:    794
MeSH:     ['Alismataceae', 'Apoptosis', 'Cell Differentiation']
Content:  areole within a window stage leaf (PCD is occurring) was divided into three 
areas based on the progression of PCD; cells that will not undergo PCD (NPCD), 
cells in early stages of PCD (EPCD), and ce

ID:       21645374_chunk2
Chunk:    2
Chars:    779
MeSH:     ['Alismataceae', 'Apoptosis', 'Cell Differentiation']
Content:  transition pore (PTP) formation during PCD was indirectly examined via in vivo 
cyclosporine A (CsA) treatment. This treatment resulted in lace plant leaves 
with a significantly lower number of perfo

ID:       216453

---
## Step 2: Ingest into Vector Databases

Same chunk corpus pushed into multiple databases for fair cross-DB comparison.

In [30]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
print(f"Using embedding model: {config.EMBEDDING_MODELS[embedding_key]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using embedding model: sentence-transformers/all-MiniLM-L6-v2


### ChromaDB

In [32]:
chroma_store = ingest_to_chroma(
    documents=documents, ids=ids, embeddings=embeddings,
    db_name=f"{embedding_key}_pubmed_chromadb"
)

/content/vectorstores/minilm_pubmed_chromadb
Total: 2849 chunks ingested into ChromaDB


### Qdrant

In [33]:
qdrant_store = ingest_to_qdrant(
    documents, ids, embeddings,
    collection_name=f"{embedding_key}_pubmed_qdrant"
)

Ingested 2849 chunks into Qdrant


### LanceDB

In [34]:
lance_db, lance_table = ingest_to_lancedb(
    documents, ids, embeddings,
    table_name=f"{embedding_key}_pubmed_lance"
)

/tmp/ipykernel_9662/668128584.py:64: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if table_name not in db.table_names():


Ingested 2849 chunks into LanceDB


## FAISS

In [35]:
faiss_store = ingest_to_faiss(
    documents, embeddings,
    index_name=f"{embedding_key}_pubmed_faiss"
)

Ingested 2849 chunks into FAISS, saved to /content/vectorstores/minilm_pubmed_faiss


### Weaviate Cloud

Requires `WEAVIATE_KEY` in `.env`.

In [36]:
weaviate_store, weaviate_client = ingest_to_weaviate(documents, ids, embeddings, f"{embedding_key}_pubmed_weaviate")

Ingested 2849 chunks into Weaviate


### Verify Ingestion Counts

In [37]:
collection = weaviate_client.collections.get(f"{embedding_key}_pubmed_weaviate")
print(f"ChromaDB: {chroma_store._collection.count()} chunks")
print(f"FAISS:    {len(faiss_store.index_to_docstore_id)} chunks")
print(f"LanceDB:  {lance_table.count_rows()} chunks")
print(f"Weaviate: {collection.aggregate.over_all().total_count} chunks")
print(f"\nAll stores ingested from {len(documents)} source documents")

ChromaDB: 2849 chunks
FAISS:    2849 chunks
LanceDB:  2849 chunks
Weaviate: 2849 chunks

All stores ingested from 2849 source documents


In [38]:
import shutil
# Moves from Colab local storage to Drive
shutil.copytree('/content/vectorstores', '/content/drive/MyDrive/LJMU/vectorstores', dirs_exist_ok=True)

'/content/drive/MyDrive/LJMU/vectorstores'

In [ ]:
import shutil

shutil.rmtree('/content/vectorstores/')